# KNN Dataset Builder

This notebook only builds the final KNN-ready dataset and exports `data/knn_dataset.csv`.


In [166]:
from importlib import util
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
DATA_INTAKE_PATH = PROJECT_ROOT / "src" / "python" / "data_intake.py"

spec = util.spec_from_file_location("project_data_intake", DATA_INTAKE_PATH)
data_intake = util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
sys.modules[spec.name] = data_intake
spec.loader.exec_module(data_intake)

normalize_dataframe = data_intake.normalize_dataframe
NA_TOKENS = data_intake.NA_TOKENS

checkpoint_windows = {
    "H1_15": (1, 15),
    "H1_30": (16, 30),
    "H1_45": (31, 45),
    "H2_15": (46, 60),
    "H2_30": (61, 75),
    "H2_45": (76, 90),
    "ET1_15": (91, 105),
    "ET2_15": (106, 120),
}
minute_window_order = list(checkpoint_windows.keys())
period_offsets = {
    "half_1": 0,
    "half_2": 45,
    "extra_time_1": 90,
    "extra_time_2": 105,
}


def load_normalized_table(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / filename, na_values=NA_TOKENS)
    return normalize_dataframe(df)


def map_minute_to_window(minute_value: float | int | None) -> str | None:
    if pd.isna(minute_value):
        return None
    minute_int = int(minute_value)
    for window_label, (start_minute, end_minute) in checkpoint_windows.items():
        if start_minute <= minute_int <= end_minute:
            return window_label
    return None


def map_absolute_minute_to_checkpoint(minute_abs: int) -> str | None:
    for checkpoint_label, (start_minute, end_minute) in checkpoint_windows.items():
        if start_minute <= minute_abs <= end_minute:
            return checkpoint_label
    return None


In [167]:
players_df = load_normalized_table("players_quarters_final.csv")
pass_df = load_normalized_table("player_appearance_pass.csv")

clean_df = players_df.copy()
clean_df["checkpoint_cont_min"] = clean_df["checkpoint"].astype(str).map(
    {label: end_minute for label, (_, end_minute) in checkpoint_windows.items()}
)
clean_df["minutes_in_game"] = (
    clean_df["checkpoint_cont_min"] - clean_df["minute_in"] + 1
).clip(lower=0).astype("int64")
clean_df["minute_in_window"] = pd.Categorical(
    clean_df["minute_in"].map(map_minute_to_window),
    categories=minute_window_order,
    ordered=True,
)

pass_df["minute_abs"] = pass_df["minute"] + pass_df["period"].map(period_offsets).astype("int64")
pass_df["checkpoint"] = pass_df["minute_abs"].map(map_absolute_minute_to_checkpoint)
pass_df = pass_df[pass_df["checkpoint"].isin(clean_df["checkpoint"].astype(str).unique())].copy()

passed_counts = (
    pass_df.groupby(["player_appearance_id", "checkpoint"], dropna=False)
    .agg(
        last15_pass_passed=("id", "size"),
        last15_pass_passed_accurate=("accurate", "sum"),
    )
    .reset_index()
)

received_counts = (
    pass_df.dropna(subset=["addressee_player_appearance_id"])
    .assign(addressee_player_appearance_id=lambda x: x["addressee_player_appearance_id"].astype("int64"))
    .groupby(["addressee_player_appearance_id", "checkpoint"], dropna=False)
    .agg(
        last15_pass_received=("id", "size"),
        last15_pass_received_accurate=("accurate", "sum"),
    )
    .reset_index()
    .rename(columns={"addressee_player_appearance_id": "player_appearance_id"})
)

clean_df = clean_df.merge(passed_counts, on=["player_appearance_id", "checkpoint"], how="left")
clean_df = clean_df.merge(received_counts, on=["player_appearance_id", "checkpoint"], how="left")

pass_feature_columns = [
    "last15_pass_received",
    "last15_pass_passed",
    "last15_pass_received_accurate",
    "last15_pass_passed_accurate",
]
for column in pass_feature_columns:
    clean_df[column] = clean_df[column].fillna(0).astype("int64")

checkpoint_order = [
    checkpoint_label
    for checkpoint_label in minute_window_order
    if checkpoint_label in set(clean_df["checkpoint"].astype(str))
]
cumulative_pass_feature_map = {
    "last15_pass_passed": "cumul_pass_passed",
    "last15_pass_passed_accurate": "cumul_pass_passed_accurate",
    "last15_pass_received": "cumul_pass_received",
    "last15_pass_received_accurate": "cumul_pass_received_accurate",
}

clean_df_with_order = clean_df.assign(
    _row_order=range(len(clean_df)),
    _checkpoint_order=pd.Categorical(
        clean_df["checkpoint"].astype(str),
        categories=checkpoint_order,
        ordered=True,
    ),
).sort_values(["player_appearance_id", "_checkpoint_order", "_row_order"])

cumulative_values = clean_df_with_order[list(cumulative_pass_feature_map.keys())].groupby(
    clean_df_with_order["player_appearance_id"]
).cumsum()
cumulative_values = cumulative_values.rename(columns=cumulative_pass_feature_map)

clean_df = pd.concat([clean_df_with_order, cumulative_values], axis=1).sort_values("_row_order")
clean_df = clean_df.drop(columns=["_row_order", "_checkpoint_order", "checkpoint_cont_min"])

clean_df["scored_after_eval_key"] = (
    clean_df["fixture_id"].astype("string")
    + "__"
    + clean_df["checkpoint"].astype("string")
    + "__"
    + clean_df["player_id"].astype("string")
)

knn_df = clean_df[
    [
        "player_id",
        "checkpoint",
        "position",
        "is_home",
        "formation",
        "subbed",
        "jersey_number",
        "last15_sprints",
        "last15_hsr",
        "last15_shots",
        "last15_shots_on_target",
        "last15_shots_under_press",
        "last15_shots_top_third",
        "cumul_sprints",
        "cumul_hsr",
        "cumul_shots",
        "cumul_shots_on_target",
        "cumul_shots_under_press",
        "cumul_shots_top_third",
        "scored_after",
        "minutes_in_game",
        "minute_in_window",
        "last15_pass_passed",
        "last15_pass_passed_accurate",
        "last15_pass_received",
        "last15_pass_received_accurate",
        "cumul_pass_passed",
        "cumul_pass_passed_accurate",
        "cumul_pass_received",
        "cumul_pass_received_accurate",
        "scored_after_eval_key",
    ]
].copy()

duplicate_eval_keys = int(knn_df["scored_after_eval_key"].duplicated().sum())
if duplicate_eval_keys != 0:
    raise ValueError(f"Duplicate scored_after_eval_key values found: {duplicate_eval_keys}")


In [168]:
knn_df.head()

,player_id,checkpoint,position,is_home,formation,subbed,jersey_number,last15_sprints,last15_hsr,last15_shots,last15_shots_on_target,last15_shots_under_press,last15_shots_top_third,cumul_sprints,cumul_hsr,cumul_shots,cumul_shots_on_target,cumul_shots_under_press,cumul_shots_top_third,scored_after,minutes_in_game,minute_in_window,last15_pass_passed,last15_pass_passed_accurate,last15_pass_received,last15_pass_received_accurate,cumul_pass_passed,cumul_pass_passed_accurate,cumul_pass_received,cumul_pass_received_accurate,scored_after_eval_key
0,2702,H1_15,G,True,4-1-4-1,False,1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,H1_15,2,2,4,2,2,2,4,2,1159__H1_15__2702
1,2732,H1_15,D,True,4-1-4-1,False,2,2,5,0,0,0,0,2,5,0,0,0,0,0,15,H1_15,7,5,7,4,7,5,7,4,1159__H1_15__2732
2,3296,H1_15,D,True,4-1-4-1,False,4,2,4,0,0,0,0,2,4,0,0,0,0,0,15,H1_15,7,7,7,6,7,7,7,6,1159__H1_15__3296
3,3297,H1_15,D,True,4-1-4-1,False,5,1,5,0,0,0,0,1,5,0,0,0,0,0,15,H1_15,4,3,4,3,4,3,4,3,1159__H1_15__3297
4,3298,H1_15,A,True,4-1-4-1,True,6,4,15,0,0,0,0,4,15,0,0,0,0,0,15,H1_15,0,0,1,1,0,0,1,1,1159__H1_15__3298


In [169]:
export_path = DATA_DIR / "knn_dataset.csv"
knn_df.to_csv(export_path, index=False)

print(f"Saved knn_dataset.csv to: {export_path}")
print(f"shape = {knn_df.shape}")
print("columns:")
print(knn_df.columns.tolist())
knn_df.head()


Saved knn_dataset.csv to: /Users/jan/Documents/competitions/hackatons/WEC_26/data/knn_dataset.csv
shape = (3486, 31)
columns:
['player_id', 'checkpoint', 'position', 'is_home', 'formation', 'subbed', 'jersey_number', 'last15_sprints', 'last15_hsr', 'last15_shots', 'last15_shots_on_target', 'last15_shots_under_press', 'last15_shots_top_third', 'cumul_sprints', 'cumul_hsr', 'cumul_shots', 'cumul_shots_on_target', 'cumul_shots_under_press', 'cumul_shots_top_third', 'scored_after', 'minutes_in_game', 'minute_in_window', 'last15_pass_passed', 'last15_pass_passed_accurate', 'last15_pass_received', 'last15_pass_received_accurate', 'cumul_pass_passed', 'cumul_pass_passed_accurate', 'cumul_pass_received', 'cumul_pass_received_accurate', 'scored_after_eval_key']


,player_id,checkpoint,position,is_home,formation,subbed,jersey_number,last15_sprints,last15_hsr,last15_shots,last15_shots_on_target,last15_shots_under_press,last15_shots_top_third,cumul_sprints,cumul_hsr,cumul_shots,cumul_shots_on_target,cumul_shots_under_press,cumul_shots_top_third,scored_after,minutes_in_game,minute_in_window,last15_pass_passed,last15_pass_passed_accurate,last15_pass_received,last15_pass_received_accurate,cumul_pass_passed,cumul_pass_passed_accurate,cumul_pass_received,cumul_pass_received_accurate,scored_after_eval_key
0,2702,H1_15,G,True,4-1-4-1,False,1,0,0,0,0,0,0,0,0,0,0,0,0,0,15,H1_15,2,2,4,2,2,2,4,2,1159__H1_15__2702
1,2732,H1_15,D,True,4-1-4-1,False,2,2,5,0,0,0,0,2,5,0,0,0,0,0,15,H1_15,7,5,7,4,7,5,7,4,1159__H1_15__2732
2,3296,H1_15,D,True,4-1-4-1,False,4,2,4,0,0,0,0,2,4,0,0,0,0,0,15,H1_15,7,7,7,6,7,7,7,6,1159__H1_15__3296
3,3297,H1_15,D,True,4-1-4-1,False,5,1,5,0,0,0,0,1,5,0,0,0,0,0,15,H1_15,4,3,4,3,4,3,4,3,1159__H1_15__3297
4,3298,H1_15,A,True,4-1-4-1,True,6,4,15,0,0,0,0,4,15,0,0,0,0,0,15,H1_15,0,0,1,1,0,0,1,1,1159__H1_15__3298
